# G2 · Medición de líneas

**Spec:** [`docs/spec_G2_codex_spectral_measurements.md`](../docs/spec_G2_codex_spectral_measurements.md)  |  **Bloque:** G · Caracterización  |  **Run de este set:** `ROXs42Bb_realigned`

Mide líneas de forma genérica (flujo, EW, centroide, FWHM, RV, límites) sobre el espectro final.

| | |
|---|---|
| **Entrada** | `spec_final_object.fits` |
| **Salida (QC/productos)** | `stages/stage_g2_qc.json` |
| **Consume aguas abajo** | G3, G4 |


## Qué hace G2 y el resultado

G2 mide las líneas espectrales de forma **genérica** (cero lógica específica de Hα — el catálogo de líneas vive en config) sobre el espectro canónico final: continuo local, flujo (directo + ajuste gaussiana⊗LSF), EW, centroide, FWHM, asimetría, RV, status detect/límite, y errores por Monte Carlo.

**Catálogo:** 24 líneas en config — Balmer (Hα/Hβ), serie de Paschen, triplete de Ca II, He I, [O I], [S II] (diagnósticos de acreción, cromosfera y outflow).

**Resultado para este objeto** — detectadas: **0** · marginales: **1** · límites superiores: **22** · no medibles: **1**.

**Reconciliación de Hα (V3):** G2 Hα = `n/d` vs E1 = `n/d` → consistente = **n/d** (`n/d` = la reconciliación no está calculada en el QC de este objeto).

**Inputs:** LSF 2.297 Å (`config.h01_lsf_fwhm_A(estimate; A4 M2 unavailable)` — ojo a la procedencia: si dice `config`, no es la medida de A4/M2), throughput 1.000 (de E4), MC n=500, **covarianza = `none`** (mientras no sea un modelo real, el MC trata los errores como independientes — salvedad). Los límites alimentan G3 (inferencia física).


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -c "from musepipe.stages.stage_g2_measure_lines import run_stage_g2; run_stage_g2('$RUN')"
```

Ligero.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_g2_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -c "from musepipe.stages.stage_g2_measure_lines import run_stage_g2; run_stage_g2(\'$RUN\')"'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_g2_qc.json', RUN_ID)
nb.show(qc, keys=['n_detected', 'n_upper_limit', 'n_not_measurable', 'halpha_reconciliation_v3', 'lsf_fwhm_A'], title='G2')


## Los términos de este QC, en físico

| Término | Qué es | Por qué importa |
|---|---|---|
| `n_detected` / `n_marginal` / `n_upper_limit` / `n_not_measurable` | El reparto del catálogo de líneas: medida, dudosa, solo cota superior, o **imposible de medir** (cae en el hueco del láser, fuera de rango o sobre una telúrica). | «No medible» no es «no hay»: distinguirlo evita convertir una laguna instrumental en un límite físico. |
| `lsf_source` / `lsf_fwhm_A` | La anchura instrumental usada para el ajuste, y de dónde sale (medida en A4/M2, no la nominal). | Una línea no resuelta tiene exactamente esta anchura: si se pone mal, el flujo integrado sale mal. |
| `covariance_used` | Si el ajuste usó la covarianza espectral de G1. | Sin ella, el error de una línea que abarca varios canales sale demasiado pequeño. |
| `throughput_applied` / `throughput_source` | La fracción de flujo que la extracción deja pasar, medida por inyección en E4. | El flujo observado se divide por ella para recuperar el intrínseco; por eso importa que no se aplique dos veces (ver D1). |
| `rv_weighted_kms` | Velocidad radial combinada de las líneas medidas. | Una línea real está a la velocidad del sistema; una a otra velocidad es sospechosa. |
| `halpha_reconciliation_v3` | Que la Hα de G2 y la de E1 cuenten lo mismo. | Dos etapas midiendo la misma línea con distinto método deben coincidir, o una de las dos está mal. |


## Resultados que llevaron a la conclusión

Resumen del catálogo y la reconciliación con E1 del `stage_g2_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('G2', 'stages/stage_g2_qc.json'):
        q = nb.load_qc('stages/stage_g2_qc.json', RUN_ID)
        print(f"catálogo: {q['catalog_n']} líneas (espectro {q['input_spectrum']['file']}, {q['input_spectrum']['method']})")
        print(f"  detectadas={q['n_detected']}  marginales={q['n_marginal']}  "
              f"límites={q['n_upper_limit']}  no_medibles={q['n_not_measurable']}")
        hr = q['halpha_reconciliation_v3']
        print(f"\nreconciliación Hα (V3): G2={hr['g2_halpha_status']} vs E1={hr['h01_verdict']} -> consistente={hr['consistent']}")
        print(f"LSF={q['lsf_fwhm_A']} Å ({q['lsf_source']}); throughput={q['throughput_applied']:.3f}; "
              f"MC n={q['mc']['n']}; covarianza={q['covariance_used']}")
        import pandas as pd
        # Tabla NATIVA de G2 (G5/characterization la copia como final_line_table.csv):
        d = pd.read_csv(nb.run_dir(RUN_ID) / 'tables' / 'g2_line_measurements.csv')
        todas = 'todas' if (d['z_score'].abs() < 5).all() else 'NO todas'
        print(f"\nz_score de las {len(d)} líneas: {d['z_score'].min():.2f} .. {d['z_score'].max():.2f} ({todas} < 5σ)")


## Plot 1 — el catálogo de líneas: ninguna detectada

El `z_score` (significancia de detección) de cada línea, de la tabla nativa de G2 (`tables/g2_line_measurements.csv`; G5 la copia como `final_line_table.csv`), coloreado por familia. Las no medibles se marcan con × gris. Hα marcada con ★.


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_g2_qc.json', RUN_ID)
    d = pd.read_csv(nb.run_dir(RUN_ID) / 'tables' / 'g2_line_measurements.csv').sort_values('rest_A')
    fams = d['family'].fillna('?').unique()
    cm = {f: c for f, c in zip(fams, plt.cm.tab10.colors)}
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (_, r) in enumerate(d.reset_index(drop=True).iterrows()):
        z = r['z_score']
        if pd.notna(z):
            ax.plot(z, i, 'o', color=cm.get(r['family'], '0.5'), ms=7)
        else:   # no medible: sin z_score, marcador distinto (no confundir con z=0)
            ax.plot(0, i, 'x', color='0.6', ms=7, mew=1.5)
        ax.text(-6.5, i, r['name'] + (' ★' if r['name'] == 'Halpha' else ''), fontsize=6.5, va='center')
    ax.axvline(5, color='tab:red', ls='--', lw=1, label='5σ detección'); ax.axvline(-5, color='tab:red', ls='--', lw=1)
    ax.axvline(0, color='0.6', lw=0.6)
    ax.set_yticks([]); ax.set_xlim(-7, 7); ax.set_xlabel('z_score (significancia de detección)')
    ax.set_title(f"G2 · {q['catalog_n']} líneas: {q['n_detected']} detectadas, "
                 f"{q['n_upper_limit']} límites, {q['n_not_measurable']} no medible(s)")
    ax.legend(fontsize=8, loc='lower right'); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g2_lines'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'zscore_forest.png', dpi=110); print('figura ->', outdir / 'zscore_forest.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — los límites de flujo 5σ por línea

El límite superior de flujo (5σ) de cada línea medible (escala log), de la tabla nativa `tables/g2_line_measurements.csv`. Es el **producto** que G3 consume (G5 la copia a `report/characterization/final_line_table.csv`); Hα (★) es la más restrictiva para el diagnóstico de acreción.


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    d = pd.read_csv(nb.run_dir(RUN_ID) / 'tables' / 'g2_line_measurements.csv')
    d = d[d['flux_upper_limit_5sigma'].notna()].sort_values('rest_A')
    x = np.arange(len(d))
    cols = ['tab:red' if n == 'Halpha' else 'tab:blue' for n in d['name']]
    fig, ax = plt.subplots(figsize=(10, 4.3))
    ax.bar(x, d['flux_upper_limit_5sigma'], color=cols)
    ax.set_yscale('log'); ax.set_xticks(x)
    ax.set_xticklabels([n + (' ★' if n == 'Halpha' else '') for n in d['name']], rotation=60, ha='right', fontsize=6.5)
    ax.set_ylabel('límite de flujo 5σ (unidad cubo)')
    ax.set_title('G2 · límites superiores de flujo por línea (Hα ★ = la más restrictiva para acreción)')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g2_lines'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'flux_limits.png', dpi=110); print('figura ->', outdir / 'flux_limits.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- 0 detectadas / 22 límites / 1 no medible(s) — reconciliación V3 con E1: consistente=n/d (G2 Hα `n/d` vs E1 `n/d`).
- Cero lógica específica de Hα: catálogo genérico de 24 líneas en config (Balmer, Paschen, Ca II, He I, [O I], [S II]).
- Salvedades de este objeto: LSF `config.h01_lsf_fwhm_A(estimate; A4 M2 unavailable)`; covarianza aplicada al MC = none.


## Conclusión (registrada)

**G2: 0 líneas detectadas, 22 límites superiores, 1 no medible(s)** en este objeto (resuelto de su `stage_g2_qc.json`).

- **Genérico:** catálogo de 24 líneas (cero lógica específica de Hα) sobre `psffit`.
- **Reconciliación:** Hα `n/d` vs E1 `n/d` → consistente=n/d.
- **Salvedades:** LSF `config.h01_lsf_fwhm_A(estimate; A4 M2 unavailable)`; covarianza aplicada = none.
- **Downstream:** los límites alimentan G3 (inferencia física, L_acc→Ṁ).
